In [1]:
# CASSANDRA setup cell — run this FIRST, in order, before any other cell.
# On Colab with GPU desired (Pythia extraction): Runtime > Change runtime
# type > GPU, THEN run this cell.
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "{repo_url}"

if IN_COLAB:
    if not Path("/content/CASSANDRA/.git").exists():
        subprocess.run(["git", "clone", REPO_URL, "/content/CASSANDRA"],
                       check=True)
    os.chdir("/content/CASSANDRA")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "en_core_web_sm"], check=True, capture_output=True)
    subprocess.run([sys.executable, "scripts/prepare_data.py",
                    "--config", "config.yaml"], check=True)
    print("Colab setup complete. Working directory:", os.getcwd())
else:
    try:
        import torch, transformer_lens, sklearn, pandas, seaborn  # noqa: F401
        print("Local environment OK. Run `make data` if data/processed "
              "is empty.")
    except ImportError as exc:
        raise SystemExit(
            f"Missing dependency ({{exc}}). Run `make setup` from the "
            "repository root before executing this notebook."
        )


/home/z/my-project/cassandra/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Local environment OK. Run `make data` if data/processed is empty.


# 02 — Activation analysis

**Objective.** Inspect the cached residual-stream activations of the statements and locate where a truth-related signal appears across layers.

**Data.** The fp16 memmap cache under `artifacts/activations/` for gpt2 (last-token and mean-pool features at the four configured layers). Probes are NOT recomputed here: the AUC heatmap reads the out-of-fold results recorded in `logs/probe_results.jsonl`.

**Method.** PCA to two components colored by label, L2 norms by layer and label, and the layer-by-domain AUC surface.

## Environment and determinism

In [2]:
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

pd.set_option("display.max_colwidth", 120)


In [3]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from cassandra_ns.report.figures import save_figure

if "google.colab" not in sys.modules:
    matplotlib.use("Agg")  # headless execution; Colab keeps its inline backend

sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 110


## Load the activation cache (gpt2)

`load_index` returns the cache index (id, layer, mode, offset) and `load_vectors` exposes one (statements x features) fp16 block at a time, so the notebook never loads the whole cache into memory. If the cache has not been built yet in this environment, the guard cell below produces it for gpt2 with the idempotent extraction CLI.

In [4]:
import subprocess
from pathlib import Path

def _ensure(artifact: str, command: list[str]) -> None:
    if not Path(artifact).exists():
        print(f"producing {artifact} via:", " ".join(command))
        subprocess.run([sys.executable, *command], check=True)

_ensure("data/processed/folds.json",
        ["scripts/prepare_data.py", "--config", "config.yaml"])


In [5]:
_ensure("artifacts/activations/gpt2_vectors.f16",
        ["scripts/extract_activations.py", "--config", "config.yaml",
         "--model", "gpt2"])

In [6]:
from cassandra_ns.config import load_config
from cassandra_ns.activations.cache import load_index, load_vectors
from cassandra_ns.data.loader import load_processed

CFG = load_config("config.yaml")
df = load_processed(CFG)
MODEL = "gpt2"
LAYERS = [m["layers"] for m in CFG["models"] if m["name"] == MODEL][0]

index = load_index(CFG, MODEL)
index.head()

,id,sha256,model,layer,mode,offset,d_model
0,animals:00000,8849b7cab9676bcd26510bc67944c6aee8dbafce79846a97dcbf8b96b84195f8,gpt2,2,last_token,0,768
1,animals:00000,8849b7cab9676bcd26510bc67944c6aee8dbafce79846a97dcbf8b96b84195f8,gpt2,2,mean_pool,768,768
2,animals:00001,4077d4a79afaa3df717de3678c1a144e44eafacb8aaf87a45667bf605e26a697,gpt2,2,last_token,1536,768
3,animals:00001,4077d4a79afaa3df717de3678c1a144e44eafacb8aaf87a45667bf605e26a697,gpt2,2,mean_pool,2304,768
4,animals:00002,e0490241261f8420ca098e1311f13c7acc2a3ac56d3a977eb84850821129f9e6,gpt2,2,last_token,3072,768


## PCA of residual activations per layer

Two principal components of the last-token activation at each configured layer, colored by true/false. Separability that grows toward middle layers and the shape it takes (a shift, a direction, a cluster split) is the geometric picture behind the linear probes.

In [7]:
from sklearn.decomposition import PCA

label_color = df["label"].map({True: "#3a7d5c", False: "#b0553a"})

fig, axes = plt.subplots(1, len(LAYERS), figsize=(4.2 * len(LAYERS), 4.0),
                         sharey=True)
for ax, layer in zip(axes, LAYERS):
    vectors = load_vectors(CFG, MODEL, layer, "last_token")
    proj = PCA(n_components=2, random_state=SEED).fit_transform(vectors)
    ax.scatter(proj[:, 0], proj[:, 1], c=label_color, s=6, alpha=0.45,
               linewidths=0)
    ax.set_title(f"layer {layer}")
    ax.set_xlabel("PC 1")
axes[0].set_ylabel("PC 2")
fig.suptitle("resid_post, last token — green true / red false")
fig.tight_layout()
save_figure(fig, "02_pca_per_layer")
plt.show()

## L2 norm by layer and label

Mean L2 norm of the activation per layer, split by label, with the spread across statements. A norm gap between classes is the simplest signal a linear probe can read; its absence does not preclude a directional signal.

In [8]:
norms = []
for layer in LAYERS:
    vectors = load_vectors(CFG, MODEL, layer, "last_token")
    norms.append(pd.DataFrame({
        "layer": layer,
        "norm": np.linalg.norm(vectors.astype(np.float32), axis=1),
        "label": df["label"].values,
    }))
norms = pd.concat(norms, ignore_index=True)

fig, ax = plt.subplots(figsize=(8.4, 4.2))
sns.pointplot(data=norms, x="layer", y="norm", hue="label", errorbar="sd",
              palette={True: "#3a7d5c", False: "#b0553a"}, ax=ax)
ax.set_xlabel("layer")
ax.set_ylabel("L2 norm of activation")
ax.legend(title="label", labels=["true", "false"])
fig.tight_layout()
save_figure(fig, "02_l2_norm_by_layer")
plt.show()

## AUC heatmap: layer x domain

Out-of-fold AUC of the best probe configuration per (layer, domain), read directly from `logs/probe_results.jsonl`. Middle layers concentrating high AUC is the pattern reported in the literature; if the surface is flat or the maximum lands at the last layer, that is the honest shape to report.

In [9]:
from cassandra_ns.probes.evaluate import load_results

results = load_results(CFG)  # section 8(d): read logs/probe_results.jsonl
per_domain = (results[(results["model"] == MODEL) & (results["domain"].notna())]
                .groupby(["layer", "domain"])["auc"].max().unstack("domain"))

fig, ax = plt.subplots(figsize=(9, 3.6))
sns.heatmap(per_domain, annot=True, fmt=".2f", cmap="mako", ax=ax,
            vmin=0.5, vmax=0.9)
ax.set_xlabel("domain")
ax.set_ylabel("layer")
fig.tight_layout()
save_figure(fig, "02_auc_layer_by_domain")
plt.show()

## Feature modes: last token vs mean pool

Every (layer, probe type) AUC plotted once per feature mode. Points above the diagonal favor last-token features (the position where the statement closes); mean-pool features average the whole sentence and can dilute or stabilize the signal.

In [10]:
pair = (results[(results["model"] == MODEL) & results["domain"].isna()]
          .groupby(["layer", "ptype", "mode"])["auc"].max().unstack("mode"))

fig, ax = plt.subplots(figsize=(5.6, 5.6))
lim = (0.45, max(0.9, pair.values.max() + 0.02))
ax.plot(lim, lim, ls="--", c="grey", lw=1)
ax.scatter(pair["last_token"], pair["mean_pool"], s=42, c="#4c78a8")
for (layer, ptype), row in pair.iterrows():
    ax.annotate(f"L{layer}:{ptype}", (row["last_token"], row["mean_pool"]),
                fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("AUC last_token")
ax.set_ylabel("AUC mean_pool")
fig.tight_layout()
save_figure(fig, "02_feature_mode_scatter")
plt.show()

## Interpretation: where the truth signal emerges

- **Layer localization.** The PCA columns and the heatmap show together at which depth the classes pull apart; in small transformers this is typically mid-stack, consistent with the 0.70-0.85 AUC range reported for this dataset family.
- **Norm vs direction.** If the L2 curves overlap but PCA separates, the signal is directional — exactly what a linear probe reads; if the norms already split, the probe may be reading magnitude, a weaker and more style-sensitive cue.
- **Feature choice.** The scatter decides the feature mode trade-off empirically; the evaluation keeps both modes with the comparison family fixed in advance.
- **What this notebook does not show.** Geometry is descriptive: no causal claim about the model representing truth follows from separability.